# TIB P4 — formal-runtime scorer on Colab T4 (notebook v2)
Choose **Runtime → Change runtime type → T4 GPU**, then run cells in order.
Upload the matching ZIP. This notebook scores **21 stored development summaries / 3 sources**;
it never loads generation models or runs selection/test. Reference B and warning outputs remain included.

Status remains **HOLD** until actual fixtures, native/long-input probes, all21 scoring and checkpoint checks pass here.
PASS is scoped scorer readiness on these development inputs, not pilot40, model quality/admission or budget approval.
Sources/references and attribution are private research material: keep the bundle/evidence in private Drive.
Do not publish ZIP or source/reference text. Pinned weights download from official Hugging Face.


In [1]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time, zipfile
from google.colab import drive, files
SESSION_STARTED = time.monotonic()
SESSION_LIMIT_SECONDS = 11.5 * 3600
EXPECTED_ZIP_SHA256 = 'b24cf8d737d0c822305b8c2698155f8b36a59a434f08fc57de622e2d6eb0d9e4'
EXPECTED_MANIFEST_SHA256 = '69825ceb4197a63f87d9100c9f4a1e8146f7e3d31b77bcc76ff945f7d40b7396'
WORK = Path('/content/TIB_SCORER_261020_V1')
WORK.mkdir(exist_ok=True)
drive.mount('/content/drive')
STORE = Path('/content/drive/MyDrive/TIB_P4_SCORER_261020_V1')
STORE.mkdir(parents=True, exist_ok=True)
OUTPUT = STORE / 'scorer-evidence'
OUTPUT.mkdir(exist_ok=True)
BOOTSTRAP_POLICY = {'amendment_id': 'tib-scorer-bootstrap-261020-v2', 'setup_cell_sha256': 'a61541f7294164231889f87262f58326405c190d1f5173403f6d7802041ccd30', 'scorer_bundle_sha256': 'b24cf8d737d0c822305b8c2698155f8b36a59a434f08fc57de622e2d6eb0d9e4', 'scorer_metrics_and_requirements_changed': False}
(OUTPUT/'bootstrap_policy.json').write_text(json.dumps(BOOTSTRAP_POLICY,indent=2))
def bootstrap_hold(reason):
    payload = {'status':'HOLD', 'scorer_ready':False, 'T4_verified':False,
               'actual_colab_execution':True, 'reason':str(reason),
               'generation_executed':False, 'selection_start_authorized':False}
    history=OUTPUT/'bootstrap-history'
    history.mkdir(exist_ok=True)
    (history/('hold-'+str(time.time_ns())+'.json')).write_text(json.dumps(payload, indent=2))
    (OUTPUT/'bootstrap_hold.json').write_text(json.dumps(payload, indent=2))
    (OUTPUT/'scorer_ready_receipt.json').write_text(json.dumps(payload, indent=2))
def sha(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as h:
        for part in iter(lambda:h.read(4*1024*1024), b''):
            digest.update(part)
    return digest.hexdigest()
print('Private checkpoint/evidence folder:', STORE)
bootstrap_hold('New notebook session; actual scorer receipt not produced yet')


Mounted at /content/drive
Private checkpoint/evidence folder: /content/drive/MyDrive/TIB_P4_SCORER_261020_V1


## 1. Upload and verify the supplied ZIP
The notebook verifies both outer ZIP and every packaged file before installing or executing it.
The package contains no selection/test records and no generation adapters.


In [2]:
uploaded = files.upload()
matches = [Path(name) for name in uploaded if name.endswith('.zip')]
assert len(matches)==1, 'Upload exactly the supplied scorer ZIP.'
archive = matches[0]
assert sha(archive)==EXPECTED_ZIP_SHA256, 'ZIP hash mismatch; do not proceed.'
BUNDLE = WORK/'bundle'
BUNDLE.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    names = z.namelist()
    assert len(names)==len(set(names)), 'Duplicate ZIP members'
    for item in z.infolist():
        target = (BUNDLE/item.filename).resolve()
        assert target.is_relative_to(BUNDLE.resolve()), 'Unsafe ZIP path'
        assert not ((item.external_attr >> 16) & 0o170000)==0o120000, 'Symlink ZIP member'
    z.extractall(BUNDLE)
assert sha(BUNDLE/'package_manifest.json')==EXPECTED_MANIFEST_SHA256
manifest = json.loads((BUNDLE/'package_manifest.json').read_text())
assert set(names)==set(manifest['files'])|{'package_manifest.json'}
for relative, expected in manifest['files'].items():
    assert sha(BUNDLE/relative)==expected, relative
print('Verified files:', len(manifest['files']), '| Stored summaries: 21 | Sources: 3')


Saving tib-scorer-colab-runtime-261020-v1.zip to tib-scorer-colab-runtime-261020-v1.zip
Verified files: 19 | Stored summaries: 21 | Sources: 3


## 2. Install the pinned runtime into a separate Python environment
Torch CUDA130 and NumPy differ from the existing CPU receipt; this versioned amendment changes the runtime only.
Exact scorer/support pins are supplied. The runner records all installed transitive versions and binds them to checkpoints.
Downloading/replacing CUDA wheels can take time if the Colab base image differs; the notebook does not relax pins.
Bootstrap v2 bypasses ensurepip with --without-pip and uses base pip --python to install into the target venv.
Existing partial venvs are repaired without deletion. All bootstrap stderr is retained in exported logs.


In [3]:
ENV = WORK/'venv'
PYTHON = ENV/'bin/python'
RUN_ENV = dict(os.environ, USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false',
               PYTHONPATH=str(BUNDLE), HF_HUB_DISABLE_TELEMETRY='1')
assert shutil.disk_usage('/content').free >= 12*1024**3, 'Need at least 12 GiB working disk before setup.'
def execute(command, log_name):
    remaining = SESSION_LIMIT_SECONDS-(time.monotonic()-SESSION_STARTED)
    assert remaining>0, 'Session guard reached'
    log_path = OUTPUT/log_name
    try:
        with log_path.open('w') as log:
            result = subprocess.run(command, cwd=BUNDLE, env=RUN_ENV, stdout=log,
                                    stderr=subprocess.STDOUT, timeout=remaining)
        print('exit', result.returncode, '|', log_path)
        print(log_path.read_text(errors='replace')[-5000:])
        return result.returncode
    except Exception as error:
        bootstrap_hold(error)
        raise
# An unsuccessful ensurepip step can leave bin/python present. Always repair the
# venv structure without deleting it or bootstrapping pip inside the target.
created = execute([sys.executable,'-m','venv','--system-site-packages','--without-pip',str(ENV)],
                  'create-venv.log')
if created:
    bootstrap_hold('venv --without-pip failed; see create-venv.log')
assert created==0, 'HOLD: venv creation failed. Export evidence using the last cell.'
probe = "import json,sys; from pathlib import Path; assert sys.prefix != sys.base_prefix; assert Path(sys.prefix).absolute() == Path(sys.argv[1]).absolute(); print(json.dumps({'executable':sys.executable,'prefix':sys.prefix,'base_prefix':sys.base_prefix,'python':sys.version}))"
verified = execute([str(PYTHON),'-c',probe,str(ENV)],'venv-identity.log')
if verified:
    bootstrap_hold('Target interpreter is not the intended venv; see venv-identity.log')
assert verified==0, 'HOLD: target venv identity failed.'
# pip >=22.3 officially supports installing into a pip-less target interpreter.
# Use the base installer only as an installer, never as the scorer interpreter.
setup = execute([sys.executable,'-m','pip','--python',str(PYTHON),'install',
                 '-r',str(BUNDLE/'requirements-colab-t4.txt')], 'install.log')
if setup:
    bootstrap_hold('Pinned dependency installation failed; see install.log (base pip needs --python support)')
assert setup==0, 'HOLD: dependency setup failed. Export evidence using the last cell.'


exit 0 | /content/drive/MyDrive/TIB_P4_SCORER_261020_V1/scorer-evidence/create-venv.log

exit 0 | /content/drive/MyDrive/TIB_P4_SCORER_261020_V1/scorer-evidence/venv-identity.log
{"executable": "/content/TIB_SCORER_261020_V1/venv/bin/python", "prefix": "/content/TIB_SCORER_261020_V1/venv", "base_prefix": "/usr", "python": "3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]"}

exit 0 | /content/drive/MyDrive/TIB_P4_SCORER_261020_V1/scorer-evidence/install.log
ist-packages (from jinja2->torch==2.11.0+cu130->-r /content/TIB_SCORER_261020_V1/bundle/requirements-colab-t4.txt (line 6)) (3.0.4)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━

## 3. Download and validate the two pinned scorer models
Only BERTScore XLM-R layer17 and the sentence-similarity model are acquired.
If the cell was interrupted, rerun: complete manifests are verified; partial files are preserved for diagnosis.


In [4]:
ASSETS = WORK/'scorer-assets'
asset_code = """
from pathlib import Path
from experiments.scripts.tib_benchmark_scorer_assets import acquire
from experiments.scripts.tib_benchmark_scorer import validate_assets
from experiments.scripts.rq2_tib_benchmark_metrics import BERT_MODEL,BERT_REVISION,SBERT_MODEL,SBERT_REVISION
root=Path(__import__('sys').argv[1])
if not all((root/k/'asset_manifest.json').exists() for k in ('bert','sentence')):
    # acquire never replaces an existing complete manifest; mixed completion needs explicit review.
    acquire(root)
for k,m,r in [('bert',BERT_MODEL,BERT_REVISION),('sentence',SBERT_MODEL,SBERT_REVISION)]:
    validate_assets(root/k,model=m,revision=r)
print('Actual pinned scorer asset hashes verified')
"""
asset_exit=execute([str(PYTHON),'-c',asset_code,str(ASSETS)],'assets.log')
if asset_exit:
    bootstrap_hold('Scorer assets incomplete/invalid; see assets.log')
assert asset_exit==0, 'HOLD: assets. Export evidence using the last cell.'


exit 0 | /content/drive/MyDrive/TIB_P4_SCORER_261020_V1/scorer-evidence/assets.log
{"model": "FacebookAI/xlm-roberta-large", "revision": "c23d21b0620b635a76227c604d44e43a9f0ee389", "files": [["config.json", 616], ["model.safetensors", 2244817354], ["sentencepiece.bpe.model", 5069051], ["tokenizer.json", 9096718], ["tokenizer_config.json", 25]]}
{"model": "sentence-transformers/paraphrase-multilingual-mpnet-base-v2", "revision": "4328cf26390c98c5e3c738b4460a05b95f4911f5", "files": [["1_Pooling/config.json", 190], ["config.json", 723], ["config_sentence_transformers.json", 122], ["model.safetensors", 1112201288], ["modules.json", 229], ["sentence_bert_config.json", 53], ["sentencepiece.bpe.model", 5069051], ["special_tokens_map.json", 239], ["tokenizer.json", 9081518], ["tokenizer_config.json", 402]]}
{"download_bytes": 3385337579, "free_bytes": 207382335488}
Verified bert/config.json: 616 bytes
Verified bert/model.safetensors: 2244817354 bytes
Verified bert/sentencepiece.bpe.model: 5069

## 4. Run actual scorer fixtures and scorer-only E2E
Fresh scoring is the default. Set `RESUME=True` only to reuse matching scorer checkpoints from an earlier attempt.
Reuse cost is separate from original scoring cost; hash/runtime mismatch becomes HOLD.
This tests scorer checkpoint persistence/readback. It does not certify generation checkpoint recovery or cross-session survival.


In [5]:
RESUME = False
remaining_hours = min(11.5,(SESSION_LIMIT_SECONDS-(time.monotonic()-SESSION_STARTED))/3600)
command=[str(PYTHON),'-m','experiments.scripts.tib_scorer_colab_runner',
         '--bundle',str(BUNDLE),'--output',str(OUTPUT),'--assets',str(ASSETS),
         '--wall-hours',str(remaining_hours)]
if RESUME:
    command.append('--resume')
launch_ns=time.time_ns()
bootstrap_hold('Scorer subprocess started; awaiting current actual receipt')
exit_code=execute(command,'scorer-'+str(launch_ns)+'.log')
if exit_code not in (0,2):
    bootstrap_hold('Scorer subprocess failed before a complete receipt; inspect logs')
receipt_path=OUTPUT/'scorer_ready_receipt.json'
if receipt_path.exists() and exit_code in (0,2):
    current=json.loads(receipt_path.read_text())
    if int(current.get('run_id',0)) >= launch_ns:
        print(json.dumps(current,indent=2))
    else:
        bootstrap_hold('Receipt does not belong to this execution; prior PASS cannot certify it')
        print((OUTPUT/'bootstrap_hold.json').read_text())
else:
    bootstrap_hold('No actual scorer-ready receipt produced')
print('Do not infer PASS from install/download completion or local tests.')


exit 0 | /content/drive/MyDrive/TIB_P4_SCORER_261020_V1/scorer-evidence/scorer-1792458128544360421.log
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Token indices sequence length is longer than the specified maximum sequence length for this model (1800 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (640 > 128). Running this sequence through the model will result in indexing errors
Scorer score bart-control__valid-341
Scorer score bart-control__vali

## 5. Download evidence ZIP (also run after a HOLD)
Includes inputs/provenance, pinned code and manifests, actual receipts, per-output metrics/cost, logs and scorer checkpoints.
Weights/venv are omitted. Send this private ZIP back for audit. No generation/selection/final execution follows automatically.


In [6]:
from datetime import datetime, timezone
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
export=STORE/('tib-scorer-evidence-'+stamp+'.zip')
with zipfile.ZipFile(export,'w',zipfile.ZIP_DEFLATED) as z:
    for root,prefix in [(OUTPUT,'evidence'),(BUNDLE,'bundle')]:
        for path in sorted(root.rglob('*')):
            if path.is_file() and '__pycache__' not in path.parts:
                z.write(path,prefix+'/'+path.relative_to(root).as_posix())
    if 'ASSETS' in globals():
        for key in ('bert','sentence'):
            manifest=ASSETS/key/'asset_manifest.json'
            if manifest.exists():
                z.write(manifest,'scorer-assets/'+key+'/asset_manifest.json')
print('ZIP:',export,'| bytes:',export.stat().st_size,'| SHA256:',sha(export))
files.download(str(export))


ZIP: /content/drive/MyDrive/TIB_P4_SCORER_261020_V1/tib-scorer-evidence-20261020T010554Z.zip | bytes: 421816 | SHA256: 116ecb2ccce3d47689fba85c35f4036f38b03ee2bab84423be2e0e11b4065ad7


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>